<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Multi-head Attention Plus Data Loading

In [1]:
# NBVAL_IGNORE_OUTPUT
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.8.0+cu126


The complete chapter code is located in [ch03.ipynb](./ch03.ipynb).

This notebook contains the main takeaway, multihead-attention implementation (plus the data loading pipeline from chapter 2)

## Data Loader from Chapter 2

In [2]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return dataloader


with open("small-text-sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()


vocab_size = 50257
output_dim = 256
max_len = 1024
context_length = max_len


token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

max_length = 4
dataloader = create_dataloader(
    raw_text, batch_size=8, max_length=max_length, stride=max_length
)

In [3]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break

In [4]:
print(input_embeddings.shape)

torch.Size([8, 4, 256])


# Multi-head Attention from Chapter 3

## Variant A: Simple implementation

In [5]:
class CausalSelfAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out  # 128
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)  # 256, 128
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )  # context_length: 4

    def forward(self, x):
        b, n_tokens, d_in = x.shape  # torch.Size([8, 4, 256])
        print("CausalSelfAttention b:", b)
        print("CausalSelfAttention n_tokens:", n_tokens)
        print("CausalSelfAttention d_in:", d_in)

        keys = self.W_key(x)  # torch.Size([8, 4, 128])
        print("CausalSelfAttention keys:", keys.shape)
        queries = self.W_query(x)  # torch.Size([8, 4, 128])
        print("CausalSelfAttention queries:", queries.shape)
        values = self.W_value(x)  # torch.Size([8, 4, 128])
        print("CausalSelfAttention values:", values.shape)

        attn_scores = queries @ keys.transpose(
            1, 2
        )  # torch.Size([8, 4, 128]) @ torch.Size([8, 128, 4])
        print("CausalSelfAttention attn_scores:", attn_scores.shape)
        attn_scores.masked_fill_(self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)
        print("CausalSelfAttention attn_scores:", attn_scores.shape)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        print("CausalSelfAttention attn_weights:", attn_weights.shape)

        attn_weights = self.dropout(attn_weights)
        print("CausalSelfAttention attn_weights:", attn_weights.shape)

        context_vec = attn_weights @ values
        print("CausalSelfAttention context_vec:", context_vec.shape)

        return context_vec


class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                CausalSelfAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )
        self.out_proj = nn.Linear(
            d_out * num_heads, d_out * num_heads
        )  # d_out * num_heads = 256

    def forward(self, x):
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        print("MultiHeadAttentionWrapper context_vec:", context_vec.shape)
        return self.out_proj(context_vec)

In [6]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim

num_heads = 2
d_out = d_in // num_heads

print("d_in:", d_in)
print("d_out:", d_out)
print("context_length:", context_length)
print("num_heads:", num_heads)
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads)
print()

batch = input_embeddings
print(batch.shape)

print()
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

d_in: 256
d_out: 128
context_length: 4
num_heads: 2

torch.Size([8, 4, 256])

CausalSelfAttention b: 8
CausalSelfAttention n_tokens: 4
CausalSelfAttention d_in: 256
CausalSelfAttention keys: torch.Size([8, 4, 128])
CausalSelfAttention queries: torch.Size([8, 4, 128])
CausalSelfAttention values: torch.Size([8, 4, 128])
CausalSelfAttention attn_scores: torch.Size([8, 4, 4])
CausalSelfAttention attn_scores: torch.Size([8, 4, 4])
CausalSelfAttention attn_weights: torch.Size([8, 4, 4])
CausalSelfAttention attn_weights: torch.Size([8, 4, 4])
CausalSelfAttention context_vec: torch.Size([8, 4, 128])
CausalSelfAttention b: 8
CausalSelfAttention n_tokens: 4
CausalSelfAttention d_in: 256
CausalSelfAttention keys: torch.Size([8, 4, 128])
CausalSelfAttention queries: torch.Size([8, 4, 128])
CausalSelfAttention values: torch.Size([8, 4, 128])
CausalSelfAttention attn_scores: torch.Size([8, 4, 4])
CausalSelfAttention attn_scores: torch.Size([8, 4, 4])
CausalSelfAttention attn_weights: torch.Size([8, 

## Variant B: Alternative implementation

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        # Reduce the projection dim to match desired output dim
        self.head_dim = d_out // num_heads
        print("MultiHeadAttention self.head_dim:", self.head_dim)

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape  # torch.Size([8, 4, 256])
        print("MultiHeadAttention b:", b)
        print("MultiHeadAttention n_tokens:", num_tokens)
        print("MultiHeadAttention d_in:", d_in)

        keys = self.W_key(x)
        print("MultiHeadAttention keys:", keys.shape)
        queries = self.W_query(x)
        print("MultiHeadAttention queries:", queries.shape)
        values = self.W_value(x)
        print("MultiHeadAttention values:", values.shape)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        print("MultiHeadAttention keys:", keys.shape)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        print("MultiHeadAttention values:", values.shape)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        print("MultiHeadAttention queries:", queries.shape)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        print("MultiHeadAttention keys:", keys.shape)
        queries = queries.transpose(1, 2)
        print("MultiHeadAttention queries:", queries.shape)
        values = values.transpose(1, 2)
        print("MultiHeadAttention values:", values.shape)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

In [8]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim
d_out = d_in
print("d_in:", d_in)
print("d_out:", d_out)
print("context_length:", context_length)

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings
print(batch.shape)

context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

d_in: 256
d_out: 256
context_length: 4
MultiHeadAttention self.head_dim: 128
torch.Size([8, 4, 256])
MultiHeadAttention b: 8
MultiHeadAttention n_tokens: 4
MultiHeadAttention d_in: 256
MultiHeadAttention keys: torch.Size([8, 4, 256])
MultiHeadAttention queries: torch.Size([8, 4, 256])
MultiHeadAttention values: torch.Size([8, 4, 256])
MultiHeadAttention keys: torch.Size([8, 4, 2, 128])
MultiHeadAttention values: torch.Size([8, 4, 2, 128])
MultiHeadAttention queries: torch.Size([8, 4, 2, 128])
MultiHeadAttention keys: torch.Size([8, 2, 4, 128])
MultiHeadAttention queries: torch.Size([8, 2, 4, 128])
MultiHeadAttention values: torch.Size([8, 2, 4, 128])
context_vecs.shape: torch.Size([8, 4, 256])
